# Olist E-Commerce Operations & Customer Experience Analytics

## Data Cleaning & Preparation

This notebook prepares the Brazilian Olist E-Commerce dataset for operational and customer experience analysis.

### Objectives
- Validate dataset structure and data quality
- Handle missing values based on business context
- Standardize data types and date fields
- Identify and handle duplicates
- Validate relationships between datasets
- Prepare clean datasets for SQL analysis and Power BI

In [4]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [5]:
customers = pd.read_csv("../Data/olist_customers_dataset.csv")

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [6]:
geolocation = pd.read_csv("../Data/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../Data/olist_order_items_dataset.csv")
payments = pd.read_csv("../Data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../Data/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../Data/olist_orders_dataset.csv")
products = pd.read_csv("../Data/olist_products_dataset.csv")
sellers = pd.read_csv("../Data/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../Data/product_category_name_translation.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [7]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [8]:
for name, df in datasets.items():
    print("=" * 60)
    print(name.upper())
    print("=" * 60)
    print(df.columns.tolist())
    print()
    

CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

GEOLOCATION
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'produ

In [9]:
for name, df in datasets.items():
    print("=" * 60)
    print(name.upper())
    print("=" * 60)
    print(df.dtypes)
    print()

CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

GEOLOCATION
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

ORDER_ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

PAYMENTS
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

REVIEWS
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_

In [10]:
quality_summary = []

for name, df in datasets.items():
    quality_summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

quality_summary = pd.DataFrame(quality_summary)

quality_summary

,dataset,rows,columns,missing_values,duplicate_rows
0,customers,99441,5,0,0
1,geolocation,1000163,5,0,261831
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,orders,99441,8,4908,0
6,products,32951,9,2448,0
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


In [11]:
for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print("=" * 60)
        print(name.upper())
        print("=" * 60)
        print(missing)
        print()

REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64

ORDERS
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64



In [12]:
orders.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].agg(
    total_orders=("order_approved_at", "size"),
    missing_approved=("order_approved_at", lambda x: x.isna().sum()),
    missing_carrier_date=("order_delivered_carrier_date", lambda x: x.isna().sum()),
    missing_customer_date=("order_delivered_customer_date", lambda x: x.isna().sum())
)

,total_orders,missing_approved,missing_carrier_date,missing_customer_date
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


In [13]:
delivered_missing_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

delivered_missing_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [14]:
delivered_date_issues = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_approved_at"].isna() |
        orders["order_delivered_carrier_date"].isna() |
        orders["order_delivered_customer_date"].isna()
    )
]

print("Delivered orders with at least one missing lifecycle date:")
print(len(delivered_date_issues))

delivered_date_issues[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

Delivered orders with at least one missing lifecycle date:
23


,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00


### Order Lifecycle Missing Values

Missing order timestamps were evaluated against `order_status` rather than removed automatically.

Most missing delivery timestamps are expected for orders that were canceled, unavailable, processing, invoiced, approved, or still shipped at the end of the observation period.

A small number of orders marked as `delivered` also contain missing lifecycle timestamps. These records are retained because they may still be valid for sales, payment, product, or customer analysis.

**Cleaning decision:** Missing lifecycle dates will not be imputed. Records will instead be excluded only from metrics that require the unavailable timestamp, preventing artificial delivery-time estimates from being introduced into the analysis.

In [15]:
geo_exact_duplicates = geolocation[geolocation.duplicated(keep=False)]

print("Total geolocation rows:", len(geolocation))
print("Exact duplicate rows:", geolocation.duplicated().sum())
print("Rows involved in duplicate groups:", len(geo_exact_duplicates))

geo_exact_duplicates.head(10)

Total geolocation rows: 1000163
Exact duplicate rows: 261831
Rows involved in duplicate groups: 390005


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP


In [16]:
geolocation_clean = geolocation.drop_duplicates().copy()

print("Rows before cleaning:", len(geolocation))
print("Rows after removing exact duplicates:", len(geolocation_clean))
print("Rows removed:", len(geolocation) - len(geolocation_clean))
print("Remaining exact duplicates:", geolocation_clean.duplicated().sum())

Rows before cleaning: 1000163
Rows after removing exact duplicates: 738332
Rows removed: 261831
Remaining exact duplicates: 0


In [17]:
zip_counts = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix")
    .size()
    .sort_values(ascending=False)
)

print("Unique ZIP code prefixes:", zip_counts.shape[0])
print("ZIP prefixes with more than one location:", (zip_counts > 1).sum())
print("Maximum rows for a single ZIP prefix:", zip_counts.max())

zip_counts.head(10)

Unique ZIP code prefixes: 19015
ZIP prefixes with more than one location: 17823
Maximum rows for a single ZIP prefix: 779


geolocation_zip_code_prefix
38400    779
35500    751
11680    727
11740    678
36400    627
38408    621
39400    620
35162    611
37200    596
35900    589
dtype: int64

In [18]:
print("Rows in cleaned geolocation:", len(geolocation_clean))
print(
    "Unique ZIP prefixes:",
    geolocation_clean["geolocation_zip_code_prefix"].nunique()
)

Rows in cleaned geolocation: 738332
Unique ZIP prefixes: 19015


In [19]:
geolocation_zip = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix")
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        geolocation_city=("geolocation_city", lambda x: x.mode().iloc[0]),
        geolocation_state=("geolocation_state", lambda x: x.mode().iloc[0])
    )
    .reset_index()
)

print("Rows in ZIP-level table:", len(geolocation_zip))
print(
    "Unique ZIP prefixes:",
    geolocation_zip["geolocation_zip_code_prefix"].nunique()
)
print(
    "Duplicate ZIP prefixes:",
    geolocation_zip["geolocation_zip_code_prefix"].duplicated().sum()
)

geolocation_zip.head()

Rows in ZIP-level table: 19015
Unique ZIP prefixes: 19015
Duplicate ZIP prefixes: 0


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.549951,-46.634027,sao paulo,SP
1,1002,-23.548228,-46.635247,sao paulo,SP
2,1003,-23.548977,-46.635313,sao paulo,SP
3,1004,-23.549550,-46.634771,sao paulo,SP
4,1005,-23.549763,-46.636100,sao paulo,SP


### Geolocation Cleaning

The raw geolocation dataset contained 1,000,163 rows, including 261,831 exact duplicate records. Exact duplicates were removed, leaving 738,332 unique coordinate records.

Because a single ZIP-code prefix can contain multiple coordinate observations, directly joining the geolocation table to customers or sellers could create many-to-many relationships and inflate analytical results.

To create a safe geographic lookup table, the data was aggregated to one record per ZIP-code prefix:

- Median latitude and longitude were used as representative coordinates.
- The most frequent city and state were retained.
- The resulting table contains 19,015 unique ZIP-code prefixes with no duplicate keys.

This ZIP-level table will be used for geographic joins in later analysis.

In [20]:
missing_product_category = products[
    products["product_category_name"].isna()
]

print("Products with missing category:", len(missing_product_category))

missing_product_category.isna().sum()

Products with missing category: 610


product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [21]:
products_clean = products.copy()

# Fix misspelled source column names
products_clean = products_clean.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

# Preserve products with missing category instead of dropping them
products_clean["product_category_name"] = (
    products_clean["product_category_name"].fillna("unknown")
)

print(
    "Missing product categories:",
    products_clean["product_category_name"].isna().sum()
)

print(
    "Products labeled as unknown:",
    (products_clean["product_category_name"] == "unknown").sum()
)

products_clean.head()

Missing product categories: 0
Products labeled as unknown: 610


,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [22]:
product_categories = set(
    products_clean["product_category_name"].dropna().unique()
)

translated_categories = set(
    category_translation["product_category_name"].dropna().unique()
)

categories_without_translation = product_categories - translated_categories

print("Unique product categories:", len(product_categories))
print("Categories in translation table:", len(translated_categories))
print("Categories without translation:", len(categories_without_translation))

categories_without_translation

Unique product categories: 74
Categories in translation table: 71
Categories without translation: 3


{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos', 'unknown'}

In [23]:
products_clean = products_clean.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

print("Rows before translation merge:", len(products))
print("Rows after translation merge:", len(products_clean))

print(
    "Products without English category:",
    products_clean["product_category_name_english"].isna().sum()
)

Rows before translation merge: 32951
Rows after translation merge: 32951
Products without English category: 623


In [24]:
manual_category_translation = {
    "unknown": "Unknown",
    "pc_gamer": "PC Gaming",
    "portateis_cozinha_e_preparadores_de_alimentos": "Portable Kitchen & Food Preparation Appliances"
}

products_clean["product_category_name_english"] = (
    products_clean["product_category_name_english"]
    .fillna(products_clean["product_category_name"].map(manual_category_translation))
)

print(
    "Missing English categories after manual mapping:",
    products_clean["product_category_name_english"].isna().sum()
)

products_clean[
    products_clean["product_category_name"].isin(manual_category_translation.keys())
][
    ["product_category_name", "product_category_name_english"]
].drop_duplicates()

Missing English categories after manual mapping: 0


,product_category_name,product_category_name_english
105,unknown,Unknown
1628,pc_gamer,PC Gaming
5821,portateis_cozinha_e_preparadores_de_alimentos,Portable Kitchen & Food Preparation Appliances


In [25]:
print("Rows in products_clean:", len(products_clean))
print("Unique product IDs:", products_clean["product_id"].nunique())
print("Duplicate product IDs:", products_clean["product_id"].duplicated().sum())

products_clean.isna().sum()

Rows in products_clean: 32951
Unique product IDs: 32951
Duplicate product IDs: 0


product_id                         0
product_category_name              0
product_name_length              610
product_description_length       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english      0
dtype: int64

### Product Data Cleaning

The product dataset contains 32,951 unique products with no duplicate product IDs.

During cleaning:

- Source column names containing the misspelling `lenght` were renamed to `length`.
- 610 products had no category or descriptive metadata. These products were retained to avoid excluding valid transactions and were assigned the category `Unknown`.
- Missing descriptive attributes such as product name length, description length, and photo quantity were preserved rather than imputed.
- Missing physical measurements were also retained because assigning artificial averages or zero values could distort later analysis.
- Portuguese product categories were mapped to English using the provided category translation table.
- Two categories missing from the translation table were manually assigned readable English labels:
  - `pc_gamer` → `PC Gaming`
  - `portateis_cozinha_e_preparadores_de_alimentos` → `Portable Kitchen & Food Preparation Appliances`
- The cleaned product table retains all 32,951 products with unique product IDs and complete English category labels.

## Orders Data Cleaning

The orders dataset contains the core order lifecycle information used to evaluate fulfillment performance, delivery reliability, and customer experience.

In [26]:
orders_clean = orders.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors="coerce"
    )

orders_clean[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [27]:
orders_clean[date_columns].isna().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [28]:
orders_clean["delivery_time_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

print(
    "Orders with delivery time:",
    orders_clean["delivery_time_days"].notna().sum()
)

print(
    "Orders without delivery time:",
    orders_clean["delivery_time_days"].isna().sum()
)

orders_clean["delivery_time_days"].describe()

Orders with delivery time: 96476
Orders without delivery time: 2965


count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_time_days, dtype: float64

In [29]:
negative_delivery_times = orders_clean[
    orders_clean["delivery_time_days"] < 0
]

print("Orders with negative delivery time:", len(negative_delivery_times))

Orders with negative delivery time: 0


In [30]:
orders_clean["delivery_delay_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

orders_clean["delivery_delay_days"].describe()

count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: delivery_delay_days, dtype: float64

In [31]:
orders_clean["is_late"] = pd.NA

valid_delivery = orders_clean["order_delivered_customer_date"].notna()

orders_clean.loc[
    valid_delivery,
    "is_late"
] = (
    orders_clean.loc[valid_delivery, "delivery_delay_days"] > 0
)

orders_clean["is_late"] = orders_clean["is_late"].astype("boolean")

orders_clean["is_late"].value_counts(dropna=False)

is_late
False    88649
True      7827
<NA>      2965
Name: count, dtype: Int64

In [32]:
measurable_deliveries = orders_clean["is_late"].notna().sum()
late_deliveries = orders_clean["is_late"].sum()

late_delivery_rate = (
    late_deliveries / measurable_deliveries
) * 100

print("Measurable deliveries:", measurable_deliveries)
print("Late deliveries:", late_deliveries)
print(f"Late delivery rate: {late_delivery_rate:.2f}%")

Measurable deliveries: 96476
Late deliveries: 7827
Late delivery rate: 8.11%


In [33]:
orders_clean["carrier_handoff_days"] = (
    orders_clean["order_delivered_carrier_date"]
    - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders_clean["carrier_handoff_days"].describe()

count    97658.000000
mean         3.234050
std          3.611996
min       -171.212419
25%          1.128967
50%          2.204676
75%          4.071777
max        125.775521
Name: carrier_handoff_days, dtype: float64

In [34]:
negative_handoff = orders_clean[
    orders_clean["carrier_handoff_days"] < 0
]

print("Orders with negative carrier handoff time:", len(negative_handoff))

negative_handoff[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "carrier_handoff_days"
    ]
].sort_values("carrier_handoff_days").head(10)

Orders with negative carrier handoff time: 166


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,carrier_handoff_days
25883,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,-171.212419
83321,4021cd7611d6d9ce5ffcd24817fc374f,delivered,2018-08-18 11:49:40,2018-08-18 12:10:30,2018-08-14 06:22:00,-4.227546
67844,db090a16182b263b1e896bb26c6f66cf,delivered,2018-07-13 16:14:08,2018-07-13 16:44:09,2018-07-13 13:59:00,-0.093843
65452,9711d975b961355b4b5d636857e48498,delivered,2018-06-13 15:23:50,2018-06-13 15:38:18,2018-06-13 13:15:00,-0.089468
79401,89d32b64af005178b318f76cd60f2c3c,delivered,2018-07-06 11:54:40,2018-07-06 12:11:05,2018-07-06 09:48:00,-0.087963
13969,6192897f85cb2aff6dd1fca56fddb45e,delivered,2018-07-18 13:34:22,2018-07-18 13:50:12,2018-07-18 11:33:00,-0.084282
43657,3ed559dec69ab96d9074fedaf18650eb,delivered,2018-06-15 13:20:09,2018-06-15 13:44:37,2018-06-15 11:22:00,-0.082049
4256,4e157a36ea9cf89bde6fff57a780b525,delivered,2018-08-24 14:37:50,2018-08-24 14:50:14,2018-08-24 12:43:00,-0.079745
74503,119060af27b04f5fae8b2b0e27eef7f9,delivered,2018-06-11 14:34:47,2018-06-12 09:31:54,2018-06-11 12:45:00,-0.076238
21910,e23c9dcc304042fa90d538be679fa68d,delivered,2018-06-13 11:41:23,2018-06-13 13:31:40,2018-06-13 09:53:00,-0.075266


In [35]:
orders_clean["invalid_carrier_handoff"] = (
    orders_clean["carrier_handoff_days"] < 0
)

orders_clean.loc[
    orders_clean["invalid_carrier_handoff"],
    "carrier_handoff_days"
] = np.nan

print(
    "Invalid carrier handoff records:",
    orders_clean["invalid_carrier_handoff"].sum()
)

orders_clean["carrier_handoff_days"].describe()

Invalid carrier handoff records: 166


count    97492.000000
mean         3.241404
std          3.569041
min          0.000370
25%          1.132046
50%          2.208611
75%          4.074387
max        125.775521
Name: carrier_handoff_days, dtype: float64

In [36]:
orders_clean["carrier_transit_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

orders_clean["carrier_transit_days"].describe()

count    96475.000000
mean         9.330547
std          8.760122
min        -16.096169
25%          4.099948
50%          7.099769
75%         12.029115
max        205.190972
Name: carrier_transit_days, dtype: float64

In [37]:
negative_transit = orders_clean[
    orders_clean["carrier_transit_days"] < 0
]

print("Orders with negative carrier transit time:", len(negative_transit))

negative_transit[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "carrier_transit_days"
    ]
].sort_values("carrier_transit_days").head(10)

Orders with negative carrier transit time: 23


,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,carrier_transit_days
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,2017-02-10 10:19:10,2017-03-02 17:34:26,2017-02-14 15:15:57,-16.096169
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-08-09 18:18:43,2017-08-01 21:13:01,-7.878958
45302,29941903985f944b0ffc49c479c1547d,delivered,2017-05-29 16:16:50,2017-06-09 15:07:29,2017-06-02 11:09:16,-7.165428
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-08-01 18:23:30,2017-07-26 18:09:10,-6.009954
49933,76458889992169d3135b264dc13aec67,delivered,2016-10-07 10:05:16,2016-10-26 11:43:06,2016-10-20 18:03:17,-5.735984
71227,19feb5627c41ea1b36a8e50a469b3644,delivered,2016-10-07 17:09:56,2016-10-26 11:42:05,2016-10-20 19:07:54,-5.690405
74967,d5558a097766363b8e76b38c43332e8a,delivered,2017-02-04 19:01:33,2017-02-15 08:55:26,2017-02-10 07:58:32,-5.039514
41636,b866af202be0692766081310cd4085e1,delivered,2017-01-27 14:59:17,2017-02-20 02:32:08,2017-02-15 03:53:46,-4.943310
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-28 16:57:58,2017-07-25 19:32:56,-2.892384
78556,ea1dcb4757a844d2642547797bd5feb0,delivered,2017-07-18 13:38:29,2017-07-27 19:21:31,2017-07-25 19:43:10,-1.984965


In [38]:
orders_clean["invalid_carrier_transit"] = (
    orders_clean["carrier_transit_days"] < 0
)

orders_clean.loc[
    orders_clean["invalid_carrier_transit"],
    "carrier_transit_days"
] = np.nan

print(
    "Invalid carrier transit records:",
    orders_clean["invalid_carrier_transit"].sum()
)

orders_clean["carrier_transit_days"].describe()

Invalid carrier transit records: 23


count    96452.000000
mean         9.333551
std          8.758825
min          0.000000
25%          4.101525
50%          7.100312
75%         12.030443
max        205.190972
Name: carrier_transit_days, dtype: float64

In [39]:
orders_clean.loc[
    orders_clean["order_delivered_customer_date"].notna(),
    "order_status"
].value_counts()

order_status
delivered    96470
canceled         6
Name: count, dtype: int64

In [40]:
# A delivery-performance KPI is valid only for completed deliveries
valid_delivered = (
    (orders_clean["order_status"] == "delivered")
    & orders_clean["order_delivered_customer_date"].notna()
    & orders_clean["order_estimated_delivery_date"].notna()
)

# Reset the flag
orders_clean["is_late"] = pd.Series(
    pd.NA,
    index=orders_clean.index,
    dtype="boolean"
)

# Calculate late/on-time only for valid delivered orders
orders_clean.loc[valid_delivered, "is_late"] = (
    orders_clean.loc[
        valid_delivered,
        "order_delivered_customer_date"
    ]
    >
    orders_clean.loc[
        valid_delivered,
        "order_estimated_delivery_date"
    ]
)

orders_clean["is_late"].value_counts(dropna=False)

is_late
False    88644
True      7826
<NA>      2971
Name: count, dtype: Int64

In [41]:
measurable_deliveries = orders_clean["is_late"].notna().sum()
late_deliveries = orders_clean["is_late"].sum()

late_delivery_rate = (
    late_deliveries / measurable_deliveries
) * 100

print("Measurable delivered orders:", measurable_deliveries)
print("Late deliveries:", late_deliveries)
print(f"Final late delivery rate: {late_delivery_rate:.2f}%")

Measurable delivered orders: 96470
Late deliveries: 7826
Final late delivery rate: 8.11%


In [42]:
orders_clean["approval_time_hours"] = (
    orders_clean["order_approved_at"]
    - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 3600

orders_clean["approval_time_hours"].describe()

count    99281.000000
mean        10.419094
std         26.038004
min          0.000000
25%          0.215000
50%          0.343333
75%         14.580833
max       4509.180556
Name: approval_time_hours, dtype: float64

In [43]:
orders_clean[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_time_hours"
    ]
].sort_values(
    "approval_time_hours",
    ascending=False
).head(10)

,order_id,order_status,order_purchase_timestamp,order_approved_at,approval_time_hours
47552,1612081119e8f23745698ad3367cc14b,unavailable,2016-10-05 18:06:48,2017-04-11 15:17:38,4509.180556
62293,2e5dc86c8c4aa663549caf5e31de840d,processing,2017-02-03 00:04:49,2017-04-04 10:56:48,1450.866389
4541,2e7a8482f6fb09756ca50c10d7bfc047,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,784.045556
4396,e5fa5a7210941f7d56d0208e4e071d35,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,781.028056
43697,0a5c74ccc786ced7903270de9d6c170a,unavailable,2018-01-18 23:14:36,2018-02-20 12:05:54,780.855000
96251,0a93b40850d3f4becf2f276666e01340,delivered,2018-01-20 14:24:50,2018-02-20 11:51:27,741.443611
55708,f7923db0430587601c2aef15ec4b8af4,delivered,2018-01-20 17:38:58,2018-02-20 12:05:54,738.448889
53475,490291524fddde2b31c2e6bec3d9e6da,canceled,2017-04-14 22:40:54,2017-05-13 02:45:06,676.070000
10071,809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,573.874167
83143,fdd647b689626410b725d1cce2ddf37c,processing,2017-12-04 10:09:35,2017-12-27 14:03:00,555.890278


In [44]:
print(
    "Approval > 24 hours:",
    (orders_clean["approval_time_hours"] > 24).sum()
)

print(
    "Approval > 48 hours:",
    (orders_clean["approval_time_hours"] > 48).sum()
)

print(
    "Approval > 7 days:",
    (orders_clean["approval_time_hours"] > 168).sum()
)

print(
    "Approval > 30 days:",
    (orders_clean["approval_time_hours"] > 720).sum()
)

Approval > 24 hours: 17419
Approval > 48 hours: 5166
Approval > 7 days: 81
Approval > 30 days: 7


In [45]:
orders_clean["estimated_delivery_days"] = (
    orders_clean["order_estimated_delivery_date"]
    - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders_clean["estimated_delivery_days"].describe()

count    99441.000000
mean        23.767650
std          8.832371
min          1.648993
25%         18.331690
50%         23.240370
75%         28.424861
max        155.135463
Name: estimated_delivery_days, dtype: float64

In [46]:
invalid_estimated_delivery = orders_clean[
    orders_clean["estimated_delivery_days"] <= 0
]

print(
    "Orders with invalid estimated delivery window:",
    len(invalid_estimated_delivery)
)

Orders with invalid estimated delivery window: 0


### Order Lifecycle Cleaning Summary

The orders dataset contains 99,441 orders and was prepared for fulfillment and customer experience analysis.

During cleaning and validation:

- Five order lifecycle fields were converted from text to datetime format:
  - purchase timestamp
  - approval timestamp
  - carrier handoff timestamp
  - customer delivery timestamp
  - estimated delivery timestamp

- Missing lifecycle timestamps were preserved because they largely correspond to order status. No delivery or approval dates were artificially imputed.

- Actual delivery time was calculated as the time between purchase and customer delivery.

- Delivery delay was calculated by comparing actual delivery with the estimated delivery date.

- Late-delivery performance was evaluated only for completed `delivered` orders with measurable delivery timestamps. Six canceled orders contained customer-delivery timestamps and were therefore excluded from this KPI.

- Of 96,470 measurable completed deliveries, 7,826 arrived after the estimated delivery date, producing a final late-delivery rate of 8.11%.

- Carrier handoff time was calculated from purchase to carrier handoff. 166 records contained impossible negative durations. The original timestamps were preserved, these records were flagged, and only their derived handoff durations were excluded from analysis.

- Carrier transit time was calculated from carrier handoff to customer delivery. 23 records contained negative transit durations. These records were similarly flagged while preserving the original source data.

- Order approval time was calculated in hours. No negative approval durations were identified. Extreme positive approval times were retained because they are unusual but not logically impossible.

- Estimated delivery windows were calculated from purchase to estimated delivery. No invalid or negative estimated-delivery windows were identified.

This approach preserves source-data integrity while preventing known timestamp anomalies and inappropriate order statuses from distorting operational KPIs.

## Order Items Data Cleaning

The order items dataset contains product-level transaction records and will be used to analyze sales value, freight costs, product categories, sellers, and order composition.

In [47]:
print("Rows:", len(order_items))
print("Unique orders:", order_items["order_id"].nunique())
print("Unique products:", order_items["product_id"].nunique())
print("Unique sellers:", order_items["seller_id"].nunique())

print(
    "Duplicate order-item keys:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "Exact duplicate rows:",
    order_items.duplicated().sum()
)

Rows: 112650
Unique orders: 98666
Unique products: 32951
Unique sellers: 3095
Duplicate order-item keys: 0
Exact duplicate rows: 0


In [48]:
order_items_clean = order_items.copy()

order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce"
)

print(
    "Missing shipping limit dates:",
    order_items_clean["shipping_limit_date"].isna().sum()
)

print(
    "Data type:",
    order_items_clean["shipping_limit_date"].dtype
)

Missing shipping limit dates: 0
Data type: datetime64[us]


In [49]:
order_items_clean[
    ["price", "freight_value"]
].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [50]:
print("Negative prices:", (order_items_clean["price"] < 0).sum())
print("Zero prices:", (order_items_clean["price"] == 0).sum())

print(
    "Negative freight values:",
    (order_items_clean["freight_value"] < 0).sum()
)

print(
    "Zero freight values:",
    (order_items_clean["freight_value"] == 0).sum()
)

Negative prices: 0
Zero prices: 0
Negative freight values: 0
Zero freight values: 383


In [51]:
orders_without_items = orders_clean[
    ~orders_clean["order_id"].isin(
        order_items_clean["order_id"]
    )
]

print(
    "Orders without order items:",
    len(orders_without_items)
)

orders_without_items["order_status"].value_counts()

Orders without order items: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [52]:
orphan_order_items = order_items_clean[
    ~order_items_clean["order_id"].isin(
        orders_clean["order_id"]
    )
]

print(
    "Order items without matching order:",
    len(orphan_order_items)
)

Order items without matching order: 0


In [53]:
orphan_products = order_items_clean[
    ~order_items_clean["product_id"].isin(
        products_clean["product_id"]
    )
]

orphan_sellers = order_items_clean[
    ~order_items_clean["seller_id"].isin(
        sellers["seller_id"]
    )
]

print(
    "Order items without matching product:",
    len(orphan_products)
)

print(
    "Order items without matching seller:",
    len(orphan_sellers)
)

Order items without matching product: 0
Order items without matching seller: 0


In [54]:
order_items_clean["item_total_value"] = (
    order_items_clean["price"]
    + order_items_clean["freight_value"]
)

order_items_clean[
    ["price", "freight_value", "item_total_value"]
].describe()

,price,freight_value,item_total_value
count,112650.000000,112650.000000,112650.000000
mean,120.653739,19.990320,140.644059
std,183.633928,15.806405,190.724394
min,0.850000,0.000000,6.080000
25%,39.900000,13.080000,55.220000
50%,74.990000,16.260000,92.320000
75%,134.900000,21.150000,157.937500
max,6735.000000,409.680000,6929.310000


In [55]:
order_items_clean["freight_to_price_ratio"] = (
    order_items_clean["freight_value"]
    / order_items_clean["price"]
)

order_items_clean["freight_to_price_ratio"].describe()

count    112650.000000
mean          0.320864
std           0.349894
min           0.000000
25%           0.134034
50%           0.231356
75%           0.393036
max          26.235294
Name: freight_to_price_ratio, dtype: float64

In [56]:
order_items_clean[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "freight_to_price_ratio"
    ]
].sort_values(
    "freight_to_price_ratio",
    ascending=False
).head(10)

,order_id,order_item_id,product_id,seller_id,price,freight_value,freight_to_price_ratio
87081,c5bdd8ef3c0ec420232e668302179113,2,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,0.85,22.30,26.235294
27652,3ee6513ae7ea23bdfab5b9ab60bffcb5,1,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,0.85,18.23,21.447059
48625,6e864b3f0ec71031117ad4cf46b7f2a1,1,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,0.85,18.23,21.447059
110535,fb265b2dc558a56445dfc48f8224e201,1,baf25ed4f8f70238cc87230379471454,128f9bfbe4c7d5185033914b1de3d39a,9.90,121.22,12.244444
94495,d642656598ae928a250620315d19e87e,1,b07fffe072c9adc235a35d8da7c0584d,dd533b429f380718b70ad9922c294bae,4.99,37.04,7.422846
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000
57309,8272b63d03f5f79c56e9e4120aec44ef,13,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000
57308,8272b63d03f5f79c56e9e4120aec44ef,12,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000
57307,8272b63d03f5f79c56e9e4120aec44ef,11,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000


In [57]:
items_per_order = (
    order_items_clean
    .groupby("order_id")
    .size()
)

items_per_order.describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

In [58]:
sellers_per_order = (
    order_items_clean
    .groupby("order_id")["seller_id"]
    .nunique()
)

print(
    "Orders with more than one seller:",
    (sellers_per_order > 1).sum()
)

print(
    "Maximum sellers in one order:",
    sellers_per_order.max()
)

sellers_per_order.value_counts().sort_index()

Orders with more than one seller: 1278
Maximum sellers in one order: 5


seller_id
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64

In [59]:
print("Rows:", len(order_items_clean))

print(
    "Duplicate order-item keys:",
    order_items_clean.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "Missing shipping limit dates:",
    order_items_clean["shipping_limit_date"].isna().sum()
)

print(
    "Missing prices:",
    order_items_clean["price"].isna().sum()
)

print(
    "Missing freight values:",
    order_items_clean["freight_value"].isna().sum()
)

print(
    "Missing item total values:",
    order_items_clean["item_total_value"].isna().sum()
)

print(
    "Order items without matching order:",
    (~order_items_clean["order_id"].isin(orders_clean["order_id"])).sum()
)

print(
    "Order items without matching product:",
    (~order_items_clean["product_id"].isin(products_clean["product_id"])).sum()
)

print(
    "Order items without matching seller:",
    (~order_items_clean["seller_id"].isin(sellers["seller_id"])).sum()
)

Rows: 112650
Duplicate order-item keys: 0
Missing shipping limit dates: 0
Missing prices: 0
Missing freight values: 0
Missing item total values: 0
Order items without matching order: 0
Order items without matching product: 0
Order items without matching seller: 0


### Order Items Cleaning Summary

The order-items dataset contains 112,650 item-level transaction records across 98,666 orders.

During cleaning and validation:

- The combination of `order_id` and `order_item_id` was validated as a unique item-level key, with no duplicate keys or exact duplicate rows.

- `shipping_limit_date` was converted to datetime format with no missing values introduced during conversion.

- Product prices and freight values were validated. No negative or missing values were identified, and all product prices were greater than zero.

- 383 item records contained zero freight charges. These records were retained because zero freight can represent valid free-shipping or seller-subsidized transactions rather than missing data.

- Referential integrity was validated across the transaction model:
  - Every order item has a matching order.
  - Every order item has a matching product.
  - Every order item has a matching seller.

- 775 orders in the main orders table do not contain item records. These consist of 603 unavailable, 164 canceled, 5 created, 2 invoiced, and 1 shipped order. No completed `delivered` orders were missing item records, so these orders were preserved without creating artificial transaction data.

- `item_total_value` was calculated as product price plus freight value to support transaction-value analysis while retaining price and freight as separate analytical measures.

- `freight_to_price_ratio` was calculated to measure shipping cost relative to product value. The median ratio is approximately 23.1%. Extreme ratios were investigated and retained because they were primarily associated with very low-priced products rather than invalid negative or missing financial values.

- Orders contain an average of approximately 1.14 items, with a median of 1 item and a maximum of 21 items.

- Most orders involve a single seller, but 1,278 orders contain products from multiple sellers, with a maximum of 5 sellers in one order. This relationship will be considered during later aggregation to prevent double-counting order-level metrics.

The cleaned order-items table retains all 112,650 source records while adding analytical measures and preserving valid financial and operational observations.

## Payments Data Cleaning

The payments dataset contains payment-level transaction records and will be used to analyze payment value, payment methods, installment behavior, and order-level customer spending.

Because an order can contain multiple payment records, the payment-table grain and relationship with the orders dataset must be validated before aggregation or analytical joins.

In [60]:
print("Rows:", len(payments))
print("Unique orders:", payments["order_id"].nunique())

print(
    "Exact duplicate rows:",
    payments.duplicated().sum()
)

print(
    "Duplicate order-payment sequence keys:",
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

payments.head()

Rows: 103886
Unique orders: 99440
Exact duplicate rows: 0
Duplicate order-payment sequence keys: 0


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [61]:
orders_without_payment = orders_clean[
    ~orders_clean["order_id"].isin(
        payments["order_id"]
    )
]

print(
    "Orders without payment records:",
    len(orders_without_payment)
)

orders_without_payment[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at"
    ]
]

Orders without payment records: 1


,order_id,order_status,order_purchase_timestamp,order_approved_at
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38


In [62]:
missing_payment_order_id = "bfbd0f9bdef84302105ad712db648a6c"

order_items_clean[
    order_items_clean["order_id"] == missing_payment_order_id
][
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "item_total_value"
    ]
]

,order_id,order_item_id,product_id,seller_id,price,freight_value,item_total_value
84389,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,44.99,2.83,47.82
84390,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,44.99,2.83,47.82
84391,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,44.99,2.83,47.82


In [63]:
payments_clean = payments.copy()

In [64]:
payments_clean[
    [
        "payment_sequential",
        "payment_installments",
        "payment_value"
    ]
].describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


In [65]:
print(
    "Zero payment values:",
    (payments_clean["payment_value"] == 0).sum()
)

print(
    "Zero installments:",
    (payments_clean["payment_installments"] == 0).sum()
)

print(
    "Negative payment values:",
    (payments_clean["payment_value"] < 0).sum()
)

payments_clean[
    (payments_clean["payment_value"] == 0)
    |
    (payments_clean["payment_installments"] == 0)
].sort_values(
    "payment_value"
).head(20)

Zero payment values: 9
Zero installments: 2
Negative payment values: 0


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [66]:
zero_payment_orders = payments_clean.loc[
    payments_clean["payment_value"] == 0,
    "order_id"
].unique()

payments_clean[
    payments_clean["order_id"].isin(zero_payment_orders)
].sort_values(
    ["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
33781,45ed6e85398a87c253db47c2d9f48216,1,voucher,1,21.13
11755,45ed6e85398a87c253db47c2d9f48216,2,voucher,1,50.01
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
40546,6ccb433e00daae1283ccc956189c82ae,1,credit_card,5,84.67
93478,6ccb433e00daae1283ccc956189c82ae,2,voucher,1,14.65
92318,6ccb433e00daae1283ccc956189c82ae,3,voucher,1,22.72
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
20963,8bcbe01d44d147f901cd3192671144db,1,credit_card,1,36.21


In [67]:
payment_type_summary = (
    payments_clean
    .groupby("payment_type")
    .agg(
        payment_records=("order_id", "size"),
        unique_orders=("order_id", "nunique"),
        total_payment_value=("payment_value", "sum"),
        average_payment_value=("payment_value", "mean")
    )
    .sort_values(
        "payment_records",
        ascending=False
    )
)

payment_type_summary

,payment_records,unique_orders,total_payment_value,average_payment_value
payment_type,,,,
credit_card,76795,76505,12542084.19,163.319021
boleto,19784,19784,2869361.27,145.034435
voucher,5775,3866,379436.87,65.703354
debit_card,1529,1528,217989.79,142.570170
not_defined,3,3,0.00,0.000000


In [68]:
zero_installment_orders = payments_clean.loc[
    payments_clean["payment_installments"] == 0,
    "order_id"
].unique()

payments_clean[
    payments_clean["order_id"].isin(zero_installment_orders)
].sort_values(
    ["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [69]:
payments_clean[
    payments_clean["order_id"].isin([
        "1a57108394169c0b47d8f876acc9ba2d",
        "744bade1fcf9ff3f31d860ace076d422"
    ])
].sort_values(
    ["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [70]:
payment_sequence_summary = (
    payments_clean
    .groupby("order_id")
    .agg(
        payment_records=("payment_sequential", "size"),
        min_sequence=("payment_sequential", "min"),
        max_sequence=("payment_sequential", "max")
    )
)

print(
    "Orders not starting at payment sequence 1:",
    (payment_sequence_summary["min_sequence"] != 1).sum()
)

payment_sequence_summary[
    payment_sequence_summary["min_sequence"] != 1
].head(20)

Orders not starting at payment sequence 1: 80


,payment_records,min_sequence,max_sequence
order_id,,,
00ac05fe0fc047c54418098eb64e3aaa,1,2,2
056c68d093c100017aab1f00f260705c,1,2,2
0668d086b3ae41adde3aed3dacbc8fae,1,2,2
0a7d898c6305101e69e9f5a05cd0130d,1,2,2
0d0e88a418636d40ea780339701db49c,1,2,2
0fdb6f73fab7005c84d4a43d8a93682b,1,2,2
159da9914b51de617fe80162939965c5,1,2,2
166b01d0fce75a595b92ad9e84e0843b,1,2,2
1896e8b8fd196151559b02e31e98ce89,1,2,2


In [71]:
print(
    "Orders not starting at payment sequence 1:",
    (payment_sequence_summary["min_sequence"] != 1).sum()
)

Orders not starting at payment sequence 1: 80


In [72]:
orphan_payments = payments_clean[
    ~payments_clean["order_id"].isin(
        orders_clean["order_id"]
    )
]

print(
    "Payment records without matching order:",
    len(orphan_payments)
)

print(
    "Unique payment orders without matching order:",
    orphan_payments["order_id"].nunique()
)

Payment records without matching order: 0
Unique payment orders without matching order: 0


In [73]:
payment_records_per_order = (
    payments_clean
    .groupby("order_id")
    .size()
)

print(payment_records_per_order.describe())

print(
    "\nOrders with more than one payment record:",
    (payment_records_per_order > 1).sum()
)

print(
    "Maximum payment records for one order:",
    payment_records_per_order.max()
)

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

Orders with more than one payment record: 2961
Maximum payment records for one order: 29


In [74]:
payment_methods_per_order = (
    payments_clean
    .groupby("order_id")["payment_type"]
    .nunique()
)

print(payment_methods_per_order.value_counts().sort_index())

print(
    "\nOrders using multiple payment methods:",
    (payment_methods_per_order > 1).sum()
)

print(
    "Maximum payment methods in one order:",
    payment_methods_per_order.max()
)

payment_type
1    97194
2     2246
Name: count, dtype: int64

Orders using multiple payment methods: 2246
Maximum payment methods in one order: 2


In [75]:
payments_order = (
    payments_clean
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_record_count=("payment_sequential", "size"),
        payment_method_count=("payment_type", "nunique"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

payments_order.head()

,order_id,total_payment_value,payment_record_count,payment_method_count,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3


In [76]:
print("Rows in payments_order:", len(payments_order))
print("Unique orders:", payments_order["order_id"].nunique())
print("Duplicate order IDs:", payments_order["order_id"].duplicated().sum())

print(
    "\nRaw payment total:",
    round(payments_clean["payment_value"].sum(), 2)
)

print(
    "Aggregated payment total:",
    round(payments_order["total_payment_value"].sum(), 2)
)

Rows in payments_order: 99440
Unique orders: 99440
Duplicate order IDs: 0

Raw payment total: 16008872.12
Aggregated payment total: 16008872.12


In [77]:
items_order = (
    order_items_clean
    .groupby("order_id")
    .agg(
        total_item_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        total_item_transaction_value=("item_total_value", "sum"),
        item_count=("order_item_id", "size")
    )
    .reset_index()
)

print("Rows:", len(items_order))
print("Unique orders:", items_order["order_id"].nunique())
print("Duplicate order IDs:", items_order["order_id"].duplicated().sum())

items_order.head()

Rows: 98666
Unique orders: 98666
Duplicate order IDs: 0


,order_id,total_item_price,total_freight_value,total_item_transaction_value,item_count
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04,1


In [78]:
payment_item_check = payments_order.merge(
    items_order,
    on="order_id",
    how="inner"
)

payment_item_check["payment_item_difference"] = (
    payment_item_check["total_payment_value"]
    - payment_item_check["total_item_transaction_value"]
)

print("Orders compared:", len(payment_item_check))

print(
    "Exact matches:",
    np.isclose(
        payment_item_check["payment_item_difference"],
        0,
        atol=0.01
    ).sum()
)

print(
    "Differences greater than 1 cent:",
    (
        payment_item_check["payment_item_difference"].abs() > 0.01
    ).sum()
)

print("\nDifference statistics:")
print(
    payment_item_check["payment_item_difference"].describe()
)

Orders compared: 98665
Exact matches: 98287
Differences greater than 1 cent: 378

Difference statistics:
count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: payment_item_difference, dtype: float64


In [79]:
payment_item_check = payment_item_check.merge(
    orders_clean[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

payment_item_check["has_payment_item_mismatch"] = (
    payment_item_check["payment_item_difference"].abs() > 0.01
)

mismatch_by_status = (
    payment_item_check[
        payment_item_check["has_payment_item_mismatch"]
    ]
    ["order_status"]
    .value_counts()
)

mismatch_by_status

order_status
delivered    372
shipped        3
canceled       2
invoiced       1
Name: count, dtype: int64

In [80]:
payment_item_check[
    payment_item_check["has_payment_item_mismatch"]
][[
    "order_id",
    "order_status",
    "total_payment_value",
    "total_item_transaction_value",
    "payment_item_difference",
    "payment_record_count",
    "payment_method_count"
]].sort_values(
    "payment_item_difference",
    key=abs,
    ascending=False
).head(20)

,order_id,order_status,total_payment_value,total_item_transaction_value,payment_item_difference,payment_record_count,payment_method_count
79537,ce6d150fb29ada17d2082f4847107665,delivered,1586.47,1403.66,182.81,1,1
42515,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,406.92,287.91,119.01,1,1
43434,70b742795bc441e94a44a084b6d9ce7a,delivered,578.82,466.93,111.89,1,1
58752,996c7e73600ad3723e8627ab7bef81e4,delivered,664.43,587.90,76.53,1,1
43437,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,274.84,213.15,61.69,1,1
72496,bc2c82b0ef78d2252b6176d1972db7c9,delivered,303.02,242.01,61.01,1,1
67466,af9ffff2ce6b3defd34fd4c78857a379,delivered,466.97,413.17,53.80,1,1
14628,262118ce178bb3e4590a3adcf6d62e6b,delivered,126.12,177.74,-51.62,1,1
73899,bfdb5bbb06458d600a33d61f5f287472,delivered,394.36,348.93,45.43,1,1
54305,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,293.89,254.45,39.44,1,1


In [81]:
mismatches = payment_item_check[
    payment_item_check["has_payment_item_mismatch"]
].copy()

print(
    "Payment greater than item total:",
    (mismatches["payment_item_difference"] > 0).sum()
)

print(
    "Payment lower than item total:",
    (mismatches["payment_item_difference"] < 0).sum()
)

print(
    "\nTotal net difference:",
    round(mismatches["payment_item_difference"].sum(), 2)
)

print(
    "Total absolute difference:",
    round(mismatches["payment_item_difference"].abs().sum(), 2)
)

Payment greater than item total: 290
Payment lower than item total: 88

Total net difference: 2870.83
Total absolute difference: 3269.97


### Payment Data Quality Summary

- The payments dataset contains 103,886 payment records covering 99,440 unique orders.
- The composite key (`order_id`, `payment_sequential`) is unique, with no exact duplicate payment records.
- One delivered order has no corresponding payment record. The order was retained and no payment value was imputed.
- All payment records reference valid orders; no orphan payment records were identified.
- 2,961 orders contain more than one payment record, while 2,246 orders use more than one payment method.
- A maximum of 29 payment records was observed for a single order, driven by multiple voucher records.
- Payment sequences are not always continuous: 80 orders do not begin at sequence 1. Original sequence values were preserved rather than reconstructed.
- Nine zero-value payment records and two zero-installment records were identified. These values were retained because there was insufficient evidence to infer replacement values.
- All original payment types were retained, including three `not_defined` records.
- Payments were aggregated to order level to prevent row multiplication when combining payment data with item-level transactions.
- The order-level payment table contains 99,440 unique orders with no duplicate order IDs.
- The aggregated payment total (16,008,872.12) exactly matches the raw payment total, confirming that aggregation did not lose or duplicate payment value.
- Payment totals were compared with aggregated item price plus freight for 98,665 comparable orders.
- 98,287 orders (99.62%) matched within a one-cent tolerance.
- 378 orders (0.38%) showed differences greater than one cent. Of these, 290 had payment totals above item-plus-freight totals and 88 had payment totals below them.
- The total net discrepancy was +2,870.83, while the total absolute discrepancy was 3,269.97.
- Most mismatched orders were successfully delivered (372 of 378), and the largest discrepancies were not explained by multiple payment records or mixed payment methods.
- Because the available datasets do not provide sufficient information to determine the cause of these discrepancies, the original financial values were preserved rather than adjusted.

## Reviews Data Cleaning

The reviews dataset contains customer feedback associated with orders and will be used to analyze customer satisfaction and its relationship with delivery performance.

Before analysis, review identifiers, order relationships, missing comments, review scores, and timestamps must be validated. Because an order may potentially contain more than one review record, the dataset grain will also be examined before joining reviews with order-level data.

In [82]:
reviews_clean = reviews.copy()

print("Rows:", len(reviews_clean))
print("Unique review IDs:", reviews_clean["review_id"].nunique())
print("Unique orders:", reviews_clean["order_id"].nunique())

print(
    "Duplicate review IDs:",
    reviews_clean["review_id"].duplicated().sum()
)

print(
    "Duplicate order IDs:",
    reviews_clean["order_id"].duplicated().sum()
)

print(
    "Exact duplicate rows:",
    reviews_clean.duplicated().sum()
)

Rows: 99224
Unique review IDs: 98410
Unique orders: 98673
Duplicate review IDs: 814
Duplicate order IDs: 551
Exact duplicate rows: 0


In [83]:
duplicate_review_id_rows = reviews_clean[
    reviews_clean["review_id"].duplicated(keep=False)
].sort_values(["review_id", "review_creation_date"])

print(
    "Rows involving repeated review IDs:",
    len(duplicate_review_id_rows)
)

print(
    "Unique repeated review IDs:",
    duplicate_review_id_rows["review_id"].nunique()
)

duplicate_review_id_rows.head(20)

Rows involving repeated review IDs: 1603
Unique repeated review IDs: 789


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [84]:
print(
    "Duplicate review_id + order_id combinations:",
    reviews_clean.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

print(
    "Unique review_id + order_id combinations:",
    reviews_clean[
        ["review_id", "order_id"]
    ].drop_duplicates().shape[0]
)

Duplicate review_id + order_id combinations: 0
Unique review_id + order_id combinations: 99224


In [85]:
reviews_per_order = (
    reviews_clean
    .groupby("order_id")
    .size()
)

print(reviews_per_order.value_counts().sort_index())

print(
    "\nOrders with multiple review records:",
    (reviews_per_order > 1).sum()
)

print(
    "Maximum review records for one order:",
    reviews_per_order.max()
)

1    98126
2      543
3        4
Name: count, dtype: int64

Orders with multiple review records: 547
Maximum review records for one order: 3


In [86]:
review_score_variation = (
    reviews_clean
    .groupby("order_id")["review_score"]
    .nunique()
)

print(
    "Orders with multiple different review scores:",
    (review_score_variation > 1).sum()
)

print(
    "Maximum number of different scores for one order:",
    review_score_variation.max()
)

Orders with multiple different review scores: 202
Maximum number of different scores for one order: 2


In [87]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_columns:
    reviews_clean[col] = pd.to_datetime(
        reviews_clean[col],
        errors="coerce"
    )

print(reviews_clean[review_date_columns].dtypes)

print(
    "\nMissing creation dates:",
    reviews_clean["review_creation_date"].isna().sum()
)

print(
    "Missing answer timestamps:",
    reviews_clean["review_answer_timestamp"].isna().sum()
)

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

Missing creation dates: 0
Missing answer timestamps: 0


In [88]:
print("Review score distribution:")
print(
    reviews_clean["review_score"]
    .value_counts()
    .sort_index()
)

print("\nMinimum score:", reviews_clean["review_score"].min())
print("Maximum score:", reviews_clean["review_score"].max())
print("Missing scores:", reviews_clean["review_score"].isna().sum())

print(
    "Invalid scores:",
    (~reviews_clean["review_score"].between(1, 5)).sum()
)

print(
    "Average review score:",
    round(reviews_clean["review_score"].mean(), 2)
)

Review score distribution:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Minimum score: 1
Maximum score: 5
Missing scores: 0
Invalid scores: 0
Average review score: 4.09


In [89]:
orphan_reviews = reviews_clean[
    ~reviews_clean["order_id"].isin(
        orders_clean["order_id"]
    )
]

orders_without_reviews = orders_clean[
    ~orders_clean["order_id"].isin(
        reviews_clean["order_id"]
    )
]

print(
    "Review records without matching order:",
    len(orphan_reviews)
)

print(
    "Unique review orders without matching order:",
    orphan_reviews["order_id"].nunique()
)

print(
    "\nOrders without any review:",
    len(orders_without_reviews)
)

print("\nStatus of orders without reviews:")
print(
    orders_without_reviews["order_status"]
    .value_counts()
)

Review records without matching order: 0
Unique review orders without matching order: 0

Orders without any review: 768

Status of orders without reviews:
order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64


In [90]:
print(
    "Missing review titles:",
    reviews_clean["review_comment_title"].isna().sum()
)

print(
    "Missing review messages:",
    reviews_clean["review_comment_message"].isna().sum()
)

reviews_clean["has_review_title"] = (
    reviews_clean["review_comment_title"].notna()
)

reviews_clean["has_review_comment"] = (
    reviews_clean["review_comment_message"].notna()
)

print(
    "\nReviews with title:",
    reviews_clean["has_review_title"].sum()
)

print(
    "Reviews with written comment:",
    reviews_clean["has_review_comment"].sum()
)

print(
    "\nWritten comment rate:",
    round(
        reviews_clean["has_review_comment"].mean() * 100,
        2
    ),
    "%"
)

Missing review titles: 87656
Missing review messages: 58247

Reviews with title: 11568
Reviews with written comment: 40977

Written comment rate: 41.3 %


In [91]:
reviews_clean["review_response_time_hours"] = (
    reviews_clean["review_answer_timestamp"]
    - reviews_clean["review_creation_date"]
).dt.total_seconds() / 3600

print(
    "Negative response times:",
    (reviews_clean["review_response_time_hours"] < 0).sum()
)

print(
    "Zero response times:",
    (reviews_clean["review_response_time_hours"] == 0).sum()
)

print("\nReview response time statistics:")
print(
    reviews_clean["review_response_time_hours"].describe()
)

Negative response times: 0
Zero response times: 0

Review response time statistics:
count    99224.000000
mean        75.575842
std        237.361183
min          2.141389
25%         24.116875
50%         40.198750
75%         74.485556
max      12448.781111
Name: review_response_time_hours, dtype: float64


In [92]:
orders_with_score_changes = (
    review_score_variation[
        review_score_variation > 1
    ].index
)

score_change_reviews = (
    reviews_clean[
        reviews_clean["order_id"].isin(
            orders_with_score_changes
        )
    ]
    .sort_values(
        ["order_id", "review_creation_date", "review_answer_timestamp"]
    )
)

score_change_reviews[
    [
        "order_id",
        "review_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ]
].head(20)

,order_id,review_id,review_score,review_creation_date,review_answer_timestamp
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-22,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-04,2018-03-05 17:02:00
89888,02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,2018-03-21,2018-03-22 01:32:08
17582,02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,2018-03-29,2018-03-30 03:16:19
37911,029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,2017-07-14,2017-07-17 13:58:06
55137,029863af4b968de1e5d6a82782e662f5,61fe4e7d1ae801bbe169eb67b86c6eda,4,2017-07-19,2017-07-20 12:06:11
69438,03c939fd7fd3b38f8485a0f95798f1f6,405eb2ea45e1dbe2662541ae5b47e2aa,3,2018-03-06,2018-03-06 19:50:32
8273,03c939fd7fd3b38f8485a0f95798f1f6,b04ed893318da5b863e878cd3d0511df,3,2018-03-20,2018-03-21 02:28:23
51527,03c939fd7fd3b38f8485a0f95798f1f6,f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,4,2018-03-29,2018-03-30 00:29:09
88624,03eba6d9fef8f5b3e811d4b5a7cca9cd,36ce47fb903bec726f89c65eac26dc9f,4,2018-02-23,2018-02-23 23:55:53


In [93]:
review_timestamp_duplicates = (
    reviews_clean
    .duplicated(
        subset=[
            "order_id",
            "review_creation_date",
            "review_answer_timestamp"
        ],
        keep=False
    )
)

print(
    "Rows sharing the same order + creation + answer timestamp:",
    review_timestamp_duplicates.sum()
)

print(
    "Orders affected:",
    reviews_clean.loc[
        review_timestamp_duplicates,
        "order_id"
    ].nunique()
)

Rows sharing the same order + creation + answer timestamp: 0
Orders affected: 0


In [94]:
reviews_order = (
    reviews_clean
    .sort_values(
        [
            "order_id",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    )
    .groupby("order_id")
    .agg(
        latest_review_score=("review_score", "last"),
        review_record_count=("review_id", "size"),
        average_review_score=("review_score", "mean"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max"),
        has_review_comment=("has_review_comment", "max")
    )
    .reset_index()
)

print("Rows:", len(reviews_order))
print("Unique orders:", reviews_order["order_id"].nunique())
print(
    "Duplicate order IDs:",
    reviews_order["order_id"].duplicated().sum()
)

reviews_order.head()

Rows: 98673
Unique orders: 98673
Duplicate order IDs: 0


,order_id,latest_review_score,review_record_count,average_review_score,min_review_score,max_review_score,has_review_comment
0,00010242fe8c5a6d1ba2dd792cb16214,5,1,5.0,5,5,True
1,00018f77f2f0320c557190d7a144bdd3,4,1,4.0,4,4,False
2,000229ec398224ef6ca0657da4fc703e,5,1,5.0,5,5,True
3,00024acbcdf0a6daa1e931b038114c75,4,1,4.0,4,4,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,5,1,5.0,5,5,True


In [95]:
latest_reviews_check = (
    reviews_clean
    .sort_values(
        [
            "order_id",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    )
    .groupby("order_id")
    .tail(1)
    [["order_id", "review_score"]]
    .rename(
        columns={"review_score": "expected_latest_score"}
    )
)

validation = reviews_order.merge(
    latest_reviews_check,
    on="order_id",
    how="left"
)

print(
    "Latest score mismatches:",
    (
        validation["latest_review_score"]
        != validation["expected_latest_score"]
    ).sum()
)

print(
    "Missing latest scores:",
    validation["latest_review_score"].isna().sum()
)

Latest score mismatches: 0
Missing latest scores: 0


### Reviews Cleaning Summary

- The reviews dataset contains **99,224 review records** covering **98,673 unique orders**.
- No exact duplicate rows were found.
- `review_id` is not unique by itself: **789 review IDs are repeated**, involving 1,603 rows.
- The combination of `review_id` and `order_id` is unique across all **99,224 records**, so no review records were removed.
- **547 orders** contain multiple review records, with a maximum of 3 reviews for a single order.
- **202 orders** contain more than one distinct review score, confirming that repeated reviews may represent changes in customer feedback.
- Review scores are complete and valid, ranging from **1 to 5**, with an overall average score of **4.09**.
- Review creation and answer timestamps were successfully converted to datetime with no missing values or conversion loss.
- Review chronology is valid: no answer timestamps occur before their corresponding creation timestamps.
- Review response time has a median of approximately **40.2 hours**. Large positive response-time outliers were preserved because they are chronologically valid.
- Review titles and written comments are optional. Missing text values were preserved rather than imputed.
- **40,977 reviews (41.3%)** contain a written comment.
- Boolean flags `has_review_title` and `has_review_comment` were created to support customer-experience analysis.
- All review records reference valid orders; there are **0 orphan reviews**.
- **768 orders have no review**, including **646 delivered orders**. Missing reviews are preserved and will not be interpreted as negative feedback.
- Because some orders contain multiple reviews, reviews will not be joined directly to the order-level fact table.
- An order-level `reviews_order` table was created containing **98,673 unique orders**.
- The **latest chronologically recorded review score** is used as the primary order-level customer satisfaction metric.
- The order-level table also retains the average, minimum and maximum review scores, review-record count, and comment indicator for additional analysis.
- Latest-review selection was validated with **0 score mismatches** and **0 missing latest scores**.

## Customers Data Cleaning

The customers dataset contains customer identifiers and geographic information associated with orders.

Both `customer_id` and `customer_unique_id` will be validated because they represent different analytical levels. In particular, `customer_unique_id` will be used to identify repeat customers across multiple orders.

In [96]:
customers_clean = customers.copy()

print("Rows:", len(customers_clean))

print(
    "Unique customer IDs:",
    customers_clean["customer_id"].nunique()
)

print(
    "Unique customer unique IDs:",
    customers_clean["customer_unique_id"].nunique()
)

print(
    "Duplicate customer IDs:",
    customers_clean["customer_id"].duplicated().sum()
)

print(
    "Customers appearing in multiple records:",
    (
        customers_clean["customer_unique_id"]
        .value_counts() > 1
    ).sum()
)

print(
    "Maximum records for one unique customer:",
    customers_clean["customer_unique_id"]
    .value_counts()
    .max()
)

print(
    "Missing values:",
    customers_clean.isna().sum().sum()
)

Rows: 99441
Unique customer IDs: 99441
Unique customer unique IDs: 96096
Duplicate customer IDs: 0
Customers appearing in multiple records: 2997
Maximum records for one unique customer: 17
Missing values: 0


In [97]:
orders_without_customer = orders_clean[
    ~orders_clean["customer_id"].isin(
        customers_clean["customer_id"]
    )
]

customers_without_order = customers_clean[
    ~customers_clean["customer_id"].isin(
        orders_clean["customer_id"]
    )
]

print(
    "Orders without matching customer:",
    len(orders_without_customer)
)

print(
    "Customer records without matching order:",
    len(customers_without_order)
)

Orders without matching customer: 0
Customer records without matching order: 0


In [98]:
customer_order_counts = (
    customers_clean["customer_unique_id"]
    .value_counts()
)

print("Customer record count distribution:")
print(
    customer_order_counts
    .value_counts()
    .sort_index()
)

print(
    "\nCustomers with more than one record:",
    (customer_order_counts > 1).sum()
)

print(
    "Percentage of unique customers with multiple records:",
    round(
        (customer_order_counts > 1).mean() * 100,
        2
    ),
    "%"
)

Customer record count distribution:
count
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

Customers with more than one record: 2997
Percentage of unique customers with multiple records: 3.12 %


In [99]:
print(
    "Unique customer states:",
    customers_clean["customer_state"].nunique()
)

print(
    "Unique customer cities:",
    customers_clean["customer_city"].nunique()
)

print(
    "Unique customer ZIP prefixes:",
    customers_clean["customer_zip_code_prefix"].nunique()
)

print("\nCustomer records by state:")
print(
    customers_clean["customer_state"]
    .value_counts()
    .sort_values(ascending=False)
)

print(
    "\nCustomer ZIP prefixes missing from geolocation lookup:",
    (
        ~customers_clean["customer_zip_code_prefix"]
        .isin(geolocation_zip["geolocation_zip_code_prefix"])
    ).sum()
)

Unique customer states: 27
Unique customer cities: 4119
Unique customer ZIP prefixes: 14994

Customer records by state:
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

Customer ZIP prefixes missing from geolocation lookup: 278


In [100]:
customers_missing_geo = customers_clean[
    ~customers_clean["customer_zip_code_prefix"]
    .isin(geolocation_zip["geolocation_zip_code_prefix"])
]

print(
    "Customer records missing geolocation:",
    len(customers_missing_geo)
)

print(
    "Unique missing ZIP prefixes:",
    customers_missing_geo["customer_zip_code_prefix"].nunique()
)

print("\nMissing geolocation records by state:")
print(
    customers_missing_geo["customer_state"]
    .value_counts()
)

Customer records missing geolocation: 278
Unique missing ZIP prefixes: 157

Missing geolocation records by state:
customer_state
DF    171
SP     15
RJ     13
MG     11
PR     11
GO      9
BA      9
ES      6
MA      4
CE      4
RS      4
PE      4
PI      3
PA      3
MT      2
RO      2
PB      2
RN      2
AL      1
SE      1
TO      1
Name: count, dtype: int64


In [101]:
customer_state_variation = (
    customers_clean
    .groupby("customer_unique_id")["customer_state"]
    .nunique()
)

print(
    "Unique customers appearing in multiple states:",
    (customer_state_variation > 1).sum()
)

print(
    "Maximum states associated with one customer:",
    customer_state_variation.max()
)

Unique customers appearing in multiple states: 39
Maximum states associated with one customer: 3


### Customers Cleaning Summary

- The customers dataset contains **99,441 records** with no missing values.
- `customer_id` is unique across all **99,441 records** and provides the direct relationship between customers and orders.
- There are **96,096 unique customers** based on `customer_unique_id`.
- `customer_unique_id` is the appropriate identifier for customer-level and repeat-customer analysis.
- **2,997 unique customers (3.12%)** appear in multiple customer records, with a maximum of **17 records** associated with one unique customer.
- These repeated records are not automatically interpreted as completed repeat purchases because order status must also be considered.
- Referential integrity between customers and orders is complete: there are **0 orders without a matching customer** and **0 customer records without a matching order**.
- Customer geography covers all **27 Brazilian states**, 4,119 cities, and 14,994 ZIP-code prefixes.
- **278 customer records across 157 ZIP prefixes** do not have a matching ZIP prefix in the cleaned geolocation lookup.
- These unmatched geographic records were preserved because customer city and state remain available even when latitude and longitude cannot be assigned.
- Missing geolocation coverage is concentrated in Distrito Federal (DF), which accounts for **171 of the 278 unmatched records**.
- **39 unique customers** are associated with records in more than one state, with a maximum of 3 states for one customer.
- Geographic information will therefore remain associated with the order-specific `customer_id` rather than assigning a single permanent location to each `customer_unique_id`.
- No customer records were removed or imputed during cleaning.

## Sellers Data Cleaning

The sellers dataset contains seller identifiers and geographic information.

This section validates seller uniqueness, missing values, geographic coverage, and referential integrity with the order-items dataset. Seller-level information will later support analysis of seller volume, delivery performance, and customer experience.

In [102]:
sellers_clean = sellers.copy()

print("Rows:", len(sellers_clean))

print(
    "Unique seller IDs:",
    sellers_clean["seller_id"].nunique()
)

print(
    "Duplicate seller IDs:",
    sellers_clean["seller_id"].duplicated().sum()
)

print(
    "Exact duplicate rows:",
    sellers_clean.duplicated().sum()
)

print(
    "Missing values:",
    sellers_clean.isna().sum().sum()
)

print(
    "Unique seller states:",
    sellers_clean["seller_state"].nunique()
)

print(
    "Unique seller cities:",
    sellers_clean["seller_city"].nunique()
)

print(
    "Unique seller ZIP prefixes:",
    sellers_clean["seller_zip_code_prefix"].nunique()
)

Rows: 3095
Unique seller IDs: 3095
Duplicate seller IDs: 0
Exact duplicate rows: 0
Missing values: 0
Unique seller states: 23
Unique seller cities: 611
Unique seller ZIP prefixes: 2246


In [103]:
items_without_seller = order_items_clean[
    ~order_items_clean["seller_id"]
    .isin(sellers_clean["seller_id"])
]

sellers_without_items = sellers_clean[
    ~sellers_clean["seller_id"]
    .isin(order_items_clean["seller_id"])
]

sellers_missing_geo = sellers_clean[
    ~sellers_clean["seller_zip_code_prefix"]
    .isin(geolocation_zip["geolocation_zip_code_prefix"])
]

print(
    "Order-item records without matching seller:",
    len(items_without_seller)
)

print(
    "Sellers without any order items:",
    len(sellers_without_items)
)

print(
    "Seller records missing geolocation:",
    len(sellers_missing_geo)
)

print(
    "Unique seller ZIP prefixes missing from geolocation:",
    sellers_missing_geo["seller_zip_code_prefix"].nunique()
)

Order-item records without matching seller: 0
Sellers without any order items: 0
Seller records missing geolocation: 7
Unique seller ZIP prefixes missing from geolocation: 7


In [104]:
print("Seller records by state:")
print(
    sellers_clean["seller_state"]
    .value_counts()
    .sort_values(ascending=False)
)

print("\nSellers missing geolocation:")
print(
    sellers_missing_geo[
        [
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ]
)

Seller records by state:
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

Sellers missing geolocation:
      seller_zip_code_prefix      seller_city seller_state
473                    82040         curitiba           PR
791                    91901     porto alegre           RS
1672                   72580         brasilia           DF
1931                    2285        sao paulo           SP
2182                    7412            aruja           SP
2986                   71551         brasilia           DF
3028                   37708  pocos de caldas           MG


### Sellers Cleaning Summary

- The sellers dataset contains **3,095 records** representing **3,095 unique sellers**.
- `seller_id` is unique across the dataset, with **0 duplicate seller IDs** and **0 exact duplicate rows**.
- There are **0 missing values** in the seller dataset.
- Sellers are distributed across **23 states**, 611 cities, and 2,246 ZIP-code prefixes.
- São Paulo (SP) contains the largest number of seller records, with **1,849 sellers**.
- Referential integrity with the order-items dataset is complete: **0 order-item records reference an unknown seller**.
- Every seller appears in the order-items dataset; there are **0 sellers without associated order items**.
- **7 seller records**, each representing a different ZIP prefix, do not have a matching ZIP prefix in the cleaned geolocation lookup.
- These sellers were preserved because their original city and state information remains available even when latitude and longitude cannot be assigned.
- No seller records were removed or imputed during cleaning.

## Final Data Quality Validation

Before exporting the processed datasets, final structural and referential checks are performed to ensure that cleaning operations did not unintentionally remove records or alter the intended analytical grain of each table.

In [105]:
final_row_counts = pd.DataFrame({
    "dataset": [
        "customers_clean",
        "geolocation_clean",
        "geolocation_zip",
        "order_items_clean",
        "payments_clean",
        "payments_order",
        "reviews_clean",
        "reviews_order",
        "orders_clean",
        "products_clean",
        "sellers_clean"
    ],
    "rows": [
        len(customers_clean),
        len(geolocation_clean),
        len(geolocation_zip),
        len(order_items_clean),
        len(payments_clean),
        len(payments_order),
        len(reviews_clean),
        len(reviews_order),
        len(orders_clean),
        len(products_clean),
        len(sellers_clean)
    ]
})

final_row_counts

,dataset,rows
0,customers_clean,99441
1,geolocation_clean,738332
2,geolocation_zip,19015
3,order_items_clean,112650
4,payments_clean,103886
5,payments_order,99440
6,reviews_clean,99224
7,reviews_order,98673
8,orders_clean,99441
9,products_clean,32951


In [106]:
print(
    "orders_clean duplicate order_id:",
    orders_clean["order_id"].duplicated().sum()
)

print(
    "customers_clean duplicate customer_id:",
    customers_clean["customer_id"].duplicated().sum()
)

print(
    "products_clean duplicate product_id:",
    products_clean["product_id"].duplicated().sum()
)

print(
    "sellers_clean duplicate seller_id:",
    sellers_clean["seller_id"].duplicated().sum()
)

print(
    "geolocation_zip duplicate ZIP prefix:",
    geolocation_zip["geolocation_zip_code_prefix"].duplicated().sum()
)

print(
    "payments_order duplicate order_id:",
    payments_order["order_id"].duplicated().sum()
)

print(
    "reviews_order duplicate order_id:",
    reviews_order["order_id"].duplicated().sum()
)

print(
    "order_items_clean duplicate order_id + order_item_id:",
    order_items_clean.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "payments_clean duplicate order_id + payment_sequential:",
    payments_clean.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

print(
    "reviews_clean duplicate review_id + order_id:",
    reviews_clean.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

orders_clean duplicate order_id: 0
customers_clean duplicate customer_id: 0
products_clean duplicate product_id: 0
sellers_clean duplicate seller_id: 0
geolocation_zip duplicate ZIP prefix: 0
payments_order duplicate order_id: 0
reviews_order duplicate order_id: 0
order_items_clean duplicate order_id + order_item_id: 0
payments_clean duplicate order_id + payment_sequential: 0
reviews_clean duplicate review_id + order_id: 0


In [107]:
print(
    "Orders without customer:",
    (~orders_clean["customer_id"]
     .isin(customers_clean["customer_id"])).sum()
)

print(
    "Order items without order:",
    (~order_items_clean["order_id"]
     .isin(orders_clean["order_id"])).sum()
)

print(
    "Order items without product:",
    (~order_items_clean["product_id"]
     .isin(products_clean["product_id"])).sum()
)

print(
    "Order items without seller:",
    (~order_items_clean["seller_id"]
     .isin(sellers_clean["seller_id"])).sum()
)

print(
    "Payments without order:",
    (~payments_clean["order_id"]
     .isin(orders_clean["order_id"])).sum()
)

print(
    "Reviews without order:",
    (~reviews_clean["order_id"]
     .isin(orders_clean["order_id"])).sum()
)

print(
    "Orders without payment:",
    (~orders_clean["order_id"]
     .isin(payments_order["order_id"])).sum()
)

print(
    "Orders without review:",
    (~orders_clean["order_id"]
     .isin(reviews_order["order_id"])).sum()
)

print(
    "Orders without items:",
    (~orders_clean["order_id"]
     .isin(order_items_clean["order_id"])).sum()
)

Orders without customer: 0
Order items without order: 0
Order items without product: 0
Order items without seller: 0
Payments without order: 0
Reviews without order: 0
Orders without payment: 1
Orders without review: 768
Orders without items: 775


### Final Validation Summary

- Final row counts were validated across all cleaned and aggregated datasets with no unexpected record loss.
- All primary and composite analytical keys were validated with **0 duplicate-key violations**.
- `orders_clean`, `payments_order`, and `reviews_order` maintain a strict **one-row-per-order** grain.
- `customers_clean`, `products_clean`, `sellers_clean`, and `geolocation_zip` maintain unique entity-level keys.
- `order_items_clean`, `payments_clean`, and `reviews_clean` retain their original transactional grain using validated composite keys.
- Referential integrity across the analytical model is complete:
  - **0 orders** reference an unknown customer.
  - **0 order items** reference an unknown order.
  - **0 order items** reference an unknown product.
  - **0 order items** reference an unknown seller.
  - **0 payment records** reference an unknown order.
  - **0 review records** reference an unknown order.
- Known source-data exceptions were intentionally preserved:
  - **1 order** has no payment record.
  - **768 orders** have no review record.
  - **775 orders** have no order-item record.
- These exceptions are treated as missing business-process information rather than repaired or artificially imputed.
- The processed datasets are structurally validated and ready for analytical modeling, SQL analysis, and Power BI.

## Analytical Order-Level Fact Table

To support reliable operational and customer-experience analysis, the cleaned order dataset is combined with customer attributes and order-level item, payment, and review aggregates.

The resulting table maintains a strict **one-row-per-order** grain. Aggregating the one-to-many transactional tables before joining prevents row multiplication and protects order-level KPIs from double counting.

In [108]:
orders_fact = (
    orders_clean
    .merge(
        customers_clean,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        items_order,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        payments_order,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        reviews_order,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

print("Rows:", len(orders_fact))
print("Unique orders:", orders_fact["order_id"].nunique())
print("Duplicate order IDs:", orders_fact["order_id"].duplicated().sum())
print("Columns:", orders_fact.shape[1])

Rows: 99441
Unique orders: 99441
Duplicate order IDs: 0
Columns: 35


In [109]:
print(
    "Orders missing item aggregates:",
    orders_fact["item_count"].isna().sum()
)

print(
    "Orders missing payment aggregates:",
    orders_fact["total_payment_value"].isna().sum()
)

print(
    "Orders missing review aggregates:",
    orders_fact["latest_review_score"].isna().sum()
)

print(
    "Orders missing customer_unique_id:",
    orders_fact["customer_unique_id"].isna().sum()
)

Orders missing item aggregates: 775
Orders missing payment aggregates: 1
Orders missing review aggregates: 768
Orders missing customer_unique_id: 0


In [110]:
from pathlib import Path

processed_dir = Path("../Data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

print("Processed data folder:", processed_dir.resolve())
print("Folder exists:", processed_dir.exists())

Processed data folder: /Users/siraj/Downloads/Olist-E-Commerce-Operations-Customer-Experience-Analytics-SQL-Power-BI-Python/Data/processed
Folder exists: True


In [111]:
orders_fact.to_csv(
    processed_dir / "fact_orders.csv",
    index=False
)

order_items_clean.to_csv(
    processed_dir / "fact_order_items.csv",
    index=False
)

payments_clean.to_csv(
    processed_dir / "fact_payments.csv",
    index=False
)

reviews_clean.to_csv(
    processed_dir / "fact_reviews.csv",
    index=False
)

customers_clean.to_csv(
    processed_dir / "dim_customers.csv",
    index=False
)

products_clean.to_csv(
    processed_dir / "dim_products.csv",
    index=False
)

sellers_clean.to_csv(
    processed_dir / "dim_sellers.csv",
    index=False
)

geolocation_zip.to_csv(
    processed_dir / "dim_geolocation_zip.csv",
    index=False
)

print("Processed datasets exported successfully.")

Processed datasets exported successfully.


In [112]:
exported_files = sorted(processed_dir.glob("*.csv"))

print("Number of exported CSV files:", len(exported_files))

for file in exported_files:
    print(file.name)

Number of exported CSV files: 8
dim_customers.csv
dim_geolocation_zip.csv
dim_products.csv
dim_sellers.csv
fact_order_items.csv
fact_orders.csv
fact_payments.csv
fact_reviews.csv
